In [121]:
import pandas as pd
import numpy as np

In [122]:
path = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

In [123]:
df= pd.read_csv(path)

In [124]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [125]:
df = df[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


## Question 1:
There's one column with missing values. What is it?

In [126]:
df.columns[df.isnull().sum() >0]

Index(['horsepower'], dtype='object')

## Question 2:
What's the median (50% percentile) for variable 'horsepower'?

In [127]:
df['horsepower'].median()

254.0

Prepare and split the dataset

In [128]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

n, n_val, n_test, n_train

(10000, 2000, 2000, 6000)

In [129]:
np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

In [130]:
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

In [131]:
y_train = df_train['fuel_efficiency_mpg']
y_test = df_test['fuel_efficiency_mpg']
y_val = df_val['fuel_efficiency_mpg']

In [132]:
del df_train["fuel_efficiency_mpg"]
del df_test["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]

## Question 3:
- We need to deal with missing values for the column from Q1.
- We have two options: fill it with 0 or with the mean of this variable.
- Try both options. For each, train a linear regression model without regularization using the code from the lessons.
- For computing the mean, use the training only!
- Use the validation dataset to evaluate the models and compare the RMSE of each option.
- Round the RMSE scores to 3 decimal digits using round(score, 3). This keeps the imputation difference visible in this release.

Which option gives better RMSE?

In [133]:
def train_linear_regression(X, y):
  m = len(X)

  ones = np.ones(m)
  X = np.column_stack([ones, X])

  XTX = X.T.dot(X)
  XTX_inv = np.linalg.inv(XTX)

  w_full = XTX_inv.dot(X.T).dot(y)

  return w_full[0], w_full[1:]

In [134]:
# filling missing values with 0

def prepare_X(df):
    df = df.copy()
    X = df.fillna(0).values
    return X

In [135]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)
X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
y_pred

array([31.64502134, 29.86572978, 31.58838899, ..., 30.53133298,
       30.75085098, 31.62912778])

In [136]:
#rmse
def rmse(y, y_pred):
  se = (y - y_pred) ** 2
  mse = se.mean()
  return np.sqrt(mse)

In [137]:
score_zero = rmse(y_val, y_pred).round(3)
score_zero

np.float64(2.205)

In [138]:
# filling missing values with mean
def prepare_X(df):
  df = df.copy()
  mean = df['horsepower'].mean()
  X = df.fillna(mean).values
  return X

In [139]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)
X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

In [140]:
score_mean = rmse(y_val, y_pred).round(3)
score_mean

np.float64(2.202)

## Question 4:
- Now let's train a regularized linear regression.
- For this question, fill the NAs with 0.
- Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
- Use RMSE to evaluate the model on the validation dataset.
- Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie.
- Which r gives the best RMSE?

If multiple options give the same best RMSE, select the smallest r.

In [141]:
def train_linear_regression_reg(X, y, r):
  m = len(X)
  ones = np.ones(m)
  X = np.column_stack([ones, X])

  XTX = X.T.dot(X)
  XTX = XTX + r * np.eye(X.shape[1])

  XTX_inv = np.linalg.inv(XTX)

  w_full = XTX_inv.dot(X.T).dot(y)

  return w_full[0], w_full[1:]

In [142]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare_X(df_train)

    w0, w = train_linear_regression_reg(X_train, y_train, r)

    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    print(r, round(score, 4))

0 2.2018
0.01 2.2018
0.1 2.2156
1 2.3442
5 2.4146
10 2.4268
100 2.4387


## Question 5
- We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
- Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
-For each seed, do the train/validation/test split with 60%/20%/20% distribution.
- Fill the missing values with 0 and train a model without regularization.
- For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
- What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
-Round the result to 3 decimal digits (round(std, 3))
What's the value of std?

In [143]:
def train_split_test(df, seed):
  n = len(df)

  n_test = int(n * 0.2)
  n_val = int(n * 0.2)
  n_train = n - n_test - n_val

  np.random.seed(seed)
  idx = np.arange(n)
  np.random.shuffle(idx)

  df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
  df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
  df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

  y_train = df_train["fuel_efficiency_mpg"]
  y_val = df_val["fuel_efficiency_mpg"]
  y_test = df_test["fuel_efficiency_mpg"]

  df_train = df_train.drop(columns="fuel_efficiency_mpg")
  df_val = df_val.drop(columns="fuel_efficiency_mpg")
  df_test = df_test.drop(columns="fuel_efficiency_mpg")

  return df_train, df_val, df_test, y_train, y_val, y_test

In [145]:
scores = []

for seed in range(10):
    df_train, df_val, df_test, y_train, y_val, y_test = train_split_test(df, seed)

    X_train = prepare_X(df_train)
    X_val = prepare_X(df_val)

    w0, w = train_linear_regression(X_train, y_train)

    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)
    scores.append(score)

    print("RMSE scores:", scores)
    print("Standard deviation:", round(np.std(scores), 3))

RMSE scores: [np.float64(2.235930696803099)]
Standard deviation: 0.0
RMSE scores: [np.float64(2.235930696803099), np.float64(2.1962436130061485)]
Standard deviation: 0.02
RMSE scores: [np.float64(2.235930696803099), np.float64(2.1962436130061485), np.float64(2.1587293193404067)]
Standard deviation: 0.032
RMSE scores: [np.float64(2.235930696803099), np.float64(2.1962436130061485), np.float64(2.1587293193404067), np.float64(2.2038998520065904)]
Standard deviation: 0.027
RMSE scores: [np.float64(2.235930696803099), np.float64(2.1962436130061485), np.float64(2.1587293193404067), np.float64(2.2038998520065904), np.float64(2.1986513731647075)]
Standard deviation: 0.025
RMSE scores: [np.float64(2.235930696803099), np.float64(2.1962436130061485), np.float64(2.1587293193404067), np.float64(2.2038998520065904), np.float64(2.1986513731647075), np.float64(2.241043143370583)]
Standard deviation: 0.027
RMSE scores: [np.float64(2.235930696803099), np.float64(2.1962436130061485), np.float64(2.15872931

## Question 6
- Split the dataset like previously, use seed 9.
- Combine train and validation datasets.
- Fill the missing values with 0 and train a model with r=0.001.
- What's the RMSE on the test dataset?

In [146]:
df_train, df_val, df_test, y_train, y_val, y_test = train_split_test(df, 9)

df_train_full = pd.concat([df_train, df_val], ignore_index=True)
y_train_full = pd.concat([y_train, y_val], ignore_index=True)

X_train_full = prepare_X(df_train_full)
X_test = prepare_X(df_test)

w0, w = train_linear_regression_reg(X_train_full, y_train_full, 0.001)

y_pred = w0 + X_test.dot(w)

score = rmse(y_test, y_pred)

print("RMSE:", score)

RMSE: 2.230055184871654
